In [1]:
import pandas as pd 
import numpy as np
from sklearn.preprocessing import StandardScaler ,LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

In [2]:
df = pd.read_csv("DateFruit_Dataset.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 898 entries, 0 to 897
Data columns (total 35 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   AREA           898 non-null    int64  
 1   PERIMETER      898 non-null    float64
 2   MAJOR_AXIS     898 non-null    float64
 3   MINOR_AXIS     898 non-null    float64
 4   ECCENTRICITY   898 non-null    float64
 5   EQDIASQ        898 non-null    float64
 6   SOLIDITY       898 non-null    float64
 7   CONVEX_AREA    898 non-null    int64  
 8   EXTENT         898 non-null    float64
 9   ASPECT_RATIO   898 non-null    float64
 10  ROUNDNESS      898 non-null    float64
 11  COMPACTNESS    898 non-null    float64
 12  SHAPEFACTOR_1  898 non-null    float64
 13  SHAPEFACTOR_2  898 non-null    float64
 14  SHAPEFACTOR_3  898 non-null    float64
 15  SHAPEFACTOR_4  898 non-null    float64
 16  MeanRR         898 non-null    float64
 17  MeanRG         898 non-null    float64
 18  MeanRB    

In [3]:
X = df.drop("Class",axis =1)
y = df["Class"]

le = LabelEncoder()
y = le.fit_transform(y)
y


array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,

In [4]:
 x_train , x_test,y_train,y_test = train_test_split(X ,y ,test_size =0.2,random_state=42)
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)
y_train.shape

(718,)

In [5]:
import torch 
x_train_tensor = torch.tensor(x_train_scaled , dtype = torch.float32)
y_train_tensor = torch.tensor(y_train, dtype = torch.long)

x_test_tensor = torch.tensor(x_test_scaled , dtype = torch.float32)
y_test_tensor = torch.tensor(y_test, dtype = torch.long)

In [6]:
from torch.utils.data import TensorDataset,DataLoader
train_dataset = TensorDataset(x_train_tensor,y_train_tensor )
test_dataset = TensorDataset(x_test_tensor,y_test_tensor)

train_loader= DataLoader(train_dataset,batch_size = 32,shuffle = True)
test_loader = DataLoader(test_dataset,batch_size = 32)

In [7]:
# Build model
import torch.nn as nn
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()

        self.model = nn.Sequential(
            # 1st layer
            nn.Linear(X.shape[1],64),
            nn.ReLU(),

            # 2nd layer 
            nn.Linear(64,64),
            nn.ReLU(),

            # out put layer
            nn.Linear(64,7) 
            # we dont have to write here softmoax AF bcs pytorch impicitly write for us 
        )
    def forward(self,x):
        return self.model(x)

        

In [8]:
# Training model 
model = ANN()
import torch.optim as optim
crietria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

epochs  = 100

for epoch in  range(epochs):
    model.train()

    running_loss = 0.0
    for xb ,yb in train_loader:
        optimizer.zero_grad()
        output = model(xb)
        loss = crietria(output,yb) # calculate loss
        loss.backward() # backprop.. 
        optimizer.step()  # update parameters

        running_loss += loss
    training_loss = running_loss/len(train_loader)
    print(f"Loss for epoch {epoch+1} / {epochs} : {training_loss}")


Loss for epoch 1 / 100 : 1.7029961347579956
Loss for epoch 2 / 100 : 1.0971890687942505
Loss for epoch 3 / 100 : 0.7300618290901184
Loss for epoch 4 / 100 : 0.5425018668174744
Loss for epoch 5 / 100 : 0.43895798921585083
Loss for epoch 6 / 100 : 0.3754713535308838
Loss for epoch 7 / 100 : 0.3366842567920685
Loss for epoch 8 / 100 : 0.3074164092540741
Loss for epoch 9 / 100 : 0.28441041707992554
Loss for epoch 10 / 100 : 0.25892722606658936
Loss for epoch 11 / 100 : 0.24488791823387146
Loss for epoch 12 / 100 : 0.23434078693389893
Loss for epoch 13 / 100 : 0.235146164894104
Loss for epoch 14 / 100 : 0.21146097779273987
Loss for epoch 15 / 100 : 0.2006852924823761
Loss for epoch 16 / 100 : 0.19044123589992523
Loss for epoch 17 / 100 : 0.18272528052330017
Loss for epoch 18 / 100 : 0.18210618197917938
Loss for epoch 19 / 100 : 0.16983380913734436
Loss for epoch 20 / 100 : 0.16858118772506714
Loss for epoch 21 / 100 : 0.16764192283153534
Loss for epoch 22 / 100 : 0.16102100908756256
Loss fo